# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 1: Ranking Signal Analysis** — which signals are associated with visibility, clicks, engagement, or movement?

Contract → three verification queries → five features → the leakage trap → self-check.

## Setup connect DuckDB to the warehouse (mid-panel month only)

Every verification query runs against the **`month=2026-03` partition** of `fact_content_daily_performance`  a mid-panel month, on purpose. The `_sample` table is exactly the final month (June 2026), i.e. the natural outcome window of any past→future label, so we never develop label logic there. *italicized text*

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
# fact_daily points at the MID-PANEL month partition (never the _sample).
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_sample':  f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

for name, src in TABLES.items():
    print(f'{name:12} {con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]:>12,} rows')

Paste your Hugging Face READ token (hf_...): ··········
dim_clients           104 rows
fact_daily      9,841,378 rows
fact_sample    11,694,072 rows


## 1. Contract in plain words

**① One row = one content item (page) on one calendar day.** The daily fact grain is `report_date × client_hash_id × content_hash_id`. The decision this work improves is *per page* ("which page should an editor review first"), even though the raw rows are page-days.

**② Tables.** `fact_content_daily_performance` (daily facts, partitioned by month) and `dim_clients` (one row per client with tracking start dates).

**③ Time window.** This slice is the single month **2026-03** — a mid-panel month. The full warehouse spans **2025-01-27 → 2026-06-30** (~17 months), but each client's history starts when their tracking began — an unbalanced panel.

**④ Label / proxy.** `is_declining` (built from a future window), sourced from `trend_direction` / `trend_pct`. The `trend_*` fields encode the label — **never features**.

**⑤ Deliberately excluded.** Product flags (`health_score`, `priority_score`, `action_type`) — not shipped; if rebuilt, they're a baseline to beat, never features (circular). Availability flags (`ga4_data_available`, `gsc_data_available`) — describe the *measurement*, not the page — used to *filter*, not learned from. Raw queries / URLs / titles / client names — scrambled out; never reconstruct or publish.

### Field classification (supporting detail for answers ④ and ⑤)

Every field goes in exactly one bucket. **Label / proxy never becomes a feature.**

| Bucket | Fields | Why / note |
|---|---|---|
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Grouping, joining, splitting only — pseudonyms, never learned from |
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | Observed, known before the decision point; gate on `gsc_data_available IS TRUE` |
| **Feature** | `ga4_sessions`, `ga4_pageviews`, `ga4_engaged_sessions`, `scroll_events` | Observed engagement; MUST filter on `ga4_data_available IS TRUE` (else zeros mean "no tracking", not "no engagement") |
| **Feature** | `sessions_ai` (+ breakdowns) | AI click-throughs; sparse — use as context/direction, not a solo target |
| **Label / proxy** | `is_declining` (built from a future window), and its source `trend_direction`/`trend_pct` | The thing we may predict. The `trend_*` fields encode the label — **never features** |
| **Excluded** | `ga4_data_available`, `gsc_data_available`, `client_has_gsc/ga4`, `has_gsc_access`… | Availability flags describe the *measurement*, not the page — used to *filter*, not learned from |
| **Excluded** | product flags (`health_score`, `priority_score`, `action_type`) | Not shipped. If rebuilt, they're a baseline to beat, never features (circular) |
| **Excluded** | raw queries / URLs / titles / client names | Scrambled out; never reconstruct or publish |

## 2. Three facts about the March 2026 slice

**Fact 1 — grain holds.** The unit of analysis is one page-day. Duplicates would break every aggregate.

**Fact 2 — row count and date span.** March 2026 has ~9.8M daily rows, spanning 2026-03-01 → 2026-03-31.

**Fact 3 — availability with `IS TRUE`.** GA4 zeros are only meaningful where the flag is TRUE. Only ~4% of rows carry real GA4 data.

In [7]:
# FACT 1 — grain: any page-day that repeats? Expect 0 rows (grain holds).
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""").fetchall()
print('duplicate page-days found:', len(dupes), '-> grain holds' if len(dupes)==0 else dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate page-days found: 0 -> grain holds


In [3]:
# FACT 2 — row count and date span for March 2026.
row = con.sql(f"SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()
print('march-2026 daily rows:', f'{row[0]:,}', '| date range:', row[1], '->', row[2])

# The _sample table is the FINAL month, not a random sample — proof it would leak future info:
sr = con.sql(f"SELECT MIN(report_date), MAX(report_date), COUNT(*) FROM {TABLES['fact_sample']}").fetchone()
print('_sample date range:', sr[0], '->', sr[1], '| rows:', f'{sr[2]:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march-2026 daily rows: 9,841,378 | date range: 2026-03-01 -> 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

_sample date range: 2026-06-01 -> 2026-06-30 | rows: 11,694,072


In [8]:
# FACT 3 — availability checked with IS TRUE (three-valued flag).
a = con.sql(f"""
    SELECT
      COUNT(*) AS total,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_true,
      COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false,
      COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null,
      COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_true
    FROM {TABLES['fact_daily']}
""").fetchone()
n = a[0]
print(f'total rows:         {a[0]:>10,}')
print(f'ga4 IS TRUE (real): {a[1]:>10,}   {a[1]/n*100:5.1f}%')
print(f'ga4 IS FALSE (fill):{a[2]:>10,}   {a[2]/n*100:5.1f}%')
print(f'ga4 IS NULL:        {a[3]:>10,}   {a[3]/n*100:5.1f}%')
print(f'gsc IS TRUE (real): {a[4]:>10,}   {a[4]/n*100:5.1f}%')
print('-> only a small share of rows carry real GA4; those that do are the rows we analyze\n   and the rest must NOT be read as "no engagement".')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total rows:          9,841,378
ga4 IS TRUE (real):    413,966     4.2%
ga4 IS FALSE (fill): 6,408,671    65.1%
ga4 IS NULL:         3,018,741    30.7%
gsc IS TRUE (real):  3,611,061    36.7%
-> only a small share of rows carry real GA4; those that do are the rows we analyze
   and the rest must NOT be read as "no engagement".


In [9]:
# Per-client history windows (unbalanced panel — supports contract answer ③).
c = con.sql(f"""
    SELECT
      COUNT(*) AS clients,
      COUNT(*) FILTER (WHERE gsc_data_start IS NOT NULL) AS with_gsc,
      COUNT(*) FILTER (WHERE ga4_data_start IS NOT NULL) AS with_ga4,
      MIN(gsc_data_start) AS gsc_earliest, MAX(gsc_data_start) AS gsc_latest,
      MIN(ga4_data_start) AS ga4_earliest, MAX(ga4_data_start) AS ga4_latest
    FROM {TABLES['dim_clients']}
""").fetchone()
print('clients:', c[0], '| with GSC window:', c[1], '| with GA4 window:', c[2])
print('GSC start range:', c[3], '->', c[4])
print('GA4 start range:', c[5], '->', c[6])
print('-> histories start at different dates; define per-client windows, never one flat calendar window')

clients: 104 | with GSC window: 67 | with GA4 window: 51
GSC start range: 2025-01-27 -> 2026-06-02
GA4 start range: 2025-10-29 -> 2026-06-01
-> histories start at different dates; define per-client windows, never one flat calendar window


## 3. Five features, max

Each feature is only usable at the moment it is **actually knowable**, and must respect the availability gates:

| # | Feature | Available when? | Gate |
|---|---|---|---|
| 1 | `gsc_impressions`, `gsc_clicks` | At end of the feature window, before the outcome window opens | `gsc_data_available IS TRUE` |
| 2 | `gsc_avg_position` | Label `0` means no position data — drop/flag, don't treat as rank 0 | `gsc_data_available IS TRUE` + row got impressions |
| 3 | `ga4_sessions`, `ga4_engaged_sessions` | Knowable only where real GA4 tracking ran in the feature window | `ga4_data_available IS TRUE` |
| 4 | `sessions_ai` | Knowable only where real GA4 ran; very sparse — a direction, not a solo target | `ga4_data_available IS TRUE` + volume floor |
| 5 | momentum (last-N vs prev-N impressions) | Computable only after N days of past history end; must NOT overlap the outcome window | per-client `gsc_data_start` before window start |

## 4. The trap: deliberate leakage experiment

On the **starter proxy label**, the label source is `trend_direction` (built from `trend_pct`). If `trend_pct` sneaks in as a feature the model simply memorizes the answer — ROC AUC collapses to ~1.0. Demo below, then the leaked column is **removed** from the feature list.

In [11]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv('content_refresh_anonymized.csv')
y  = (df.trend_direction == 'down').astype(int)
base_feats = ['impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days', 'word_count']
X_clean = df[base_feats].fillna(0)
X_leaky = X_clean.copy(); X_leaky['trend_pct'] = df['trend_pct'].fillna(0)  # the leak

def auc(X):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, m.predict_proba(Xte)[:, 1])

print('label base rate:', round(y.mean(), 3))
print('AUC WITHOUT leaked trend_pct:', round(auc(X_clean), 3))
print('AUC WITH leaked trend_pct      :', round(auc(X_leaky), 3))

# Remove the leak: assert the label-source columns are gone from the final feature frame.
FINAL_FEATURES = [c for c in base_feats if c not in ('trend_direction', 'trend_pct', 'trend_direction')]
print('final feature list has no trend_*:', not any('trend_' in c for c in FINAL_FEATURES))
print('final features:', FINAL_FEATURES)

label base rate: 0.542
AUC WITHOUT leaked trend_pct: 0.589
AUC WITH leaked trend_pct      : 1.0
final feature list has no trend_*: True
final features: ['impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days', 'word_count']


## 5. Data limits

One named limitation of this slice: **it is a single mid-panel month (2026-03) of an unbalanced panel, and only ~4% of its rows carry real GA4 data.**

- A GA4-based feature (engagement, AI sessions) is only valid on the `ga4_data_available IS TRUE` rows (≈4% of March), so GA4 work either filters hard to a thin slice or must be treated as exploratory.
- One month cannot capture seasonality or long persistence; it is a contract-verification slice, not the modeling population.
- This snapshot gives **proxy labels from current buckets** (`trend_direction`, `is_declining`); a true *observed future outcome* (features from prior N days → outcome over the next N days) must be built from the full panel with a strict window split, never from this one month.

In [ ]:
a2 = con.sql(f"""
    SELECT
      COUNT(*) AS total,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_real
    FROM {TABLES['fact_daily']}
""").fetchone()
print('march rows:', f'{a2[0]:,}', '| with real GA4:', f'{a2[1]:,}',
      '=', round(a2[1] / a2[0] * 100, 1), '%')
print('-> any GA4 feature must filter IS TRUE and will retain only this share of the slice')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.